# 005-E3 GLACIER pilot 28 (DEV run, not submission)

- 28 molecules from `pilot_gl_input.json` (union|single|10.0 FPNet top25 shortlists, local CPU). Each item: `{mid, cands[SMILES top25], spectra}`.
- Uses `ahmedberatozer/casmi26-glacier` (MassSpecGym MIT checkpoint + `gl_runner.py` + `gl_fuse.py` + `src/ms_pred` + pure-torch shims) and wheels from `ahmedberatozer/casmi26-iceberg` (cp312 RDKit 2025.03, subprocess-isolated via `--site`). Notebook process never imports the shim; only the `gl_runner.py` subprocess does.
- NIST weights are NOT used anywhere (see discussion/744470). Checkpoint is `ckpt/glacier.pt` from the glacier dataset (stripped MassSpecGym, sha `9192e4c2eeafe833`).
- `enable_internet=true` is DEV-only (probe/install fallback). Final submission notebooks stay offline. This run writes `glacier_scores.json` + `pilot_summary.json` to `/kaggle/working/` for retrieval.
- Budget: `budget_s=1500` for the runner + 300s grace. 28 mols, 632 cands, ~1387 unique (smiles,adduct,instr,CE) jobs expected; covered spectra ~65/136, rest unsupported (negative/unsupported adduct) counted, not scored as 0.

In [ ]:
import sys, os, glob, json, time
print('python', sys.version)
print('INPUT_ROOT', sorted(os.listdir('/kaggle/input')))
for slug in ['casmi26-glacier', 'casmi26-iceberg', 'casmi26-glacier-pilot-28']:
    print('---', slug)
    paths = sorted(glob.glob(f'/kaggle/input/{slug}/**/*', recursive=True))
    print('n_paths', len(paths))
    for p in paths[:40]:
        print(p)
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print('mem_allocated_gb', round(torch.cuda.memory_allocated()/2**30, 3))
except Exception as e:
    print('torch probe failed', repr(e)[:500])
    raise

In [ ]:
import sys, os, glob, json, time
t0 = time.time()
def find_first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    print(pattern, '->', len(hits))
    return hits
GL_CANDS = find_first('/kaggle/input/**/gl_runner.py')
assert len(GL_CANDS) >= 1, 'gl_runner.py not found under /kaggle/input: ' + str(sorted(os.listdir('/kaggle/input')))
GL_PKG = os.path.dirname(GL_CANDS[0])
print('GL_PKG', GL_PKG)
assert os.path.exists(os.path.join(GL_PKG, 'gl_fuse.py')), 'gl_fuse.py missing'
assert os.path.exists(os.path.join(GL_PKG, 'ckpt', 'glacier.pt')), 'ckpt/glacier.pt missing'
WHL = find_first('/kaggle/input/**/wheels')
ICE_WHEELS = WHL[0] if WHL else None
if ICE_WHEELS is None:
    ICE_WHEELS_CANDS = sorted(glob.glob('/kaggle/input/casmi26-iceberg/wheels'))
    assert len(ICE_WHEELS_CANDS) == 1, 'iceberg wheels not found'
    ICE_WHEELS = ICE_WHEELS_CANDS[0]
print('ICE_WHEELS', ICE_WHEELS, sorted(os.listdir(ICE_WHEELS)))
PILOT_CANDS = find_first('/kaggle/input/**/pilot_gl_input.json')
assert len(PILOT_CANDS) >= 1, 'pilot input not found'
PILOT = PILOT_CANDS[0]
print('PILOT', PILOT, round(os.path.getsize(PILOT)/2**20, 2), 'MB')
sys.path.insert(0, GL_PKG)
from gl_fuse import run_gl
items = json.load(open(PILOT))
print('n_mols', len(items), 'n_cands', sum(len(x['cands']) for x in items), 'n_spectra', sum(len(x['spectra']) for x in items))
# fail loudly if input shape is wrong (no silent 0-score run)
assert len(items) == 28, f'expected 28 pilot mols, got {len(items)}'
for x in items:
    assert isinstance(x['mid'], str) and len(x['cands']) >= 1 and len(x['spectra']) >= 1, f'bad item {x.get("mid")}'
    for s in x['cands']:
        assert isinstance(s, str) and len(s) >= 2, f'bad SMILES in {x["mid"]}'
print(f'input ok in {time.time()-t0:.1f}s')

In [ ]:
import subprocess, sys
# cp312 wheels cannot install on this 3.13 worker -> PyPI fallback (DEV, internet on).
# rdkit will be 2025.09 (NOT panel-exact 2025.03) + --any-rdkit override; recorded as deviation.
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
    'rdkit', 'einops', 'pytorch_lightning', 'lightning-utilities', 'torchmetrics'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-1500:])
import importlib
for m in ('rdkit', 'einops', 'pytorch_lightning', 'torchmetrics'):
    v = importlib.import_module(m)
    print(m, getattr(v, '__version__', 'ok'))


In [ ]:
import os, json, time, glob
t0 = time.time()
out = run_gl(GL_PKG, items, workdir='/kaggle/working/gl_work', device='cuda', budget_s=1500.0, site='none', wheels=None, extra_args=('--any-rdkit',), log=print)
print(f'run_gl returned in {time.time()-t0:.1f}s, n_mols {len(out)}')
# fail loudly on total failure (empty output means systematic error, not 0-score)
assert isinstance(out, dict) and len(out) == 28, f'gl output incomplete: {len(out) if isinstance(out, dict) else type(out)}'
with open('/kaggle/working/glacier_scores.json', 'w') as f:
    json.dump(out, f)
print('wrote /kaggle/working/glacier_scores.json', round(os.path.getsize('/kaggle/working/glacier_scores.json')/1024, 1), 'KB')
# copy runner meta alongside for timing/VRAM/counts
import shutil
meta_src = '/kaggle/working/gl_work/gl_out.json.meta.json'
assert os.path.exists(meta_src), f'runner meta missing: {glob.glob("/kaggle/working/gl_work/*")}'
shutil.copyfile(meta_src, '/kaggle/working/gl_meta.json')
meta = json.load(open(meta_src))
print(json.dumps({k: meta.get(k) for k in ('status','n_mols','n_mols_covered','n_mols_scored','n_cands','n_scored','seconds','pred_per_s','device','rdkit','max_mem_gb','ckpt_sha256','ms_pred_commit')}, indent=1))

In [ ]:
import json, os
out = json.load(open('/kaggle/working/glacier_scores.json'))
meta = json.load(open('/kaggle/working/gl_meta.json'))
n_cands = sum(len(v) for v in out.values())
n_scored = sum(1 for v in out.values() for s in v.values() if s is not None)
n_null = n_cands - n_scored
per_mol = {mid: {'n_cands': len(v), 'n_scored': sum(1 for s in v.values() if s is not None)} for mid, v in out.items()}
n_mols_scored = sum(1 for v in per_mol.values() if v['n_scored'] > 0)
n_mols_null = len(per_mol) - n_mols_scored
secs = float(meta.get('seconds', 0) or 0)
summary = {
  'n_molecules': len(out),
  'n_cands': int(n_cands),
  'n_scored': int(n_scored),
  'n_null': int(n_null),
  'n_mols_scored': int(n_mols_scored),
  'n_mols_all_null': int(n_mols_null),
  'seconds': secs,
  'sec_per_cand': (secs/max(1,n_cands)),
  'sec_per_scored': (secs/max(1,n_scored)),
  'max_mem_gb': meta.get('max_mem_gb'),
  'device': meta.get('device'),
  'rdkit': meta.get('rdkit'),
  'status': meta.get('status'),
  'pred_per_s': meta.get('pred_per_s'),
  'per_molecule': per_mol,
  'runner_meta': meta,
}
print(json.dumps({k: summary[k] for k in ('n_molecules','n_cands','n_scored','n_null','n_mols_scored','n_mols_all_null','seconds','sec_per_cand','sec_per_scored','max_mem_gb','device','rdkit','status','pred_per_s')}, indent=1))
with open('/kaggle/working/pilot_summary.json', 'w') as f:
    json.dump(summary, f, indent=1)
print('wrote /kaggle/working/pilot_summary.json + glacier_scores.json; list:', os.listdir('/kaggle/working'))